# 05 · Sharing interactive results

The question is always: **does the interaction need Python while the reader uses it?**

| Option | Needs a server? | Reader needs | Interactions that work |
| --- | --- | --- | --- |
| Standalone HTML file (Plotly, Altair, Bokeh) | No | A browser | Everything computed in JavaScript: hover, zoom, legend, Altair selections and sliders |
| Notebook exported to HTML (`nbconvert`) | No | A browser | Same as above, with your narrative around it |
| Python in the browser (WebAssembly: stlite, `panel convert`) | No (static hosting) | A modern browser, patience on first load | Most of an app, run client-side |
| Hosted app (Streamlit, Dash, Panel) | Yes | A link (and maybe a login) | Anything: callbacks, caching, live data, re-aggregation |

Start from the left. Only move right when the reader truly needs Python to run.

In [ ]:
import os
import altair as alt
import plotly.express as px

os.makedirs("exports", exist_ok=True)

def size(path):
    return f"{os.path.getsize(path) / 1e6:.2f} MB"

gap = px.data.gapminder()
g07 = gap.query("year == 2007")

## 1. Plotly -> HTML

`include_plotlyjs` decides where the ~4.5 MB plotly.js library comes from:

- `True` (default): embedded. Large, works **offline** forever.
- `"cdn"`: a `<script>` link. Tiny, needs internet when opened.

In [ ]:
fig = px.scatter(gap, x="gdpPercap", y="lifeExp", size="pop", color="continent",
                 hover_name="country", log_x=True, size_max=55,
                 animation_frame="year", animation_group="country",
                 range_x=[200, 60_000], range_y=[25, 90])

fig.write_html("exports/gapminder_offline.html", include_plotlyjs=True)
fig.write_html("exports/gapminder_cdn.html", include_plotlyjs="cdn")
print("offline:", size("exports/gapminder_offline.html"))
print("cdn:    ", size("exports/gapminder_cdn.html"))

Open both files from your file browser: they work with no Jupyter and no Python.

`fig.to_html(full_html=False, include_plotlyjs=False)` returns just a `<div>`, for pasting into a web page, blog or LMS that already loads plotly.js.

## 2. Altair -> HTML

Same trade-off: `inline=True` embeds the Vega JavaScript libraries (offline; needs the `vl-convert-python` package). Selections and bound widgets keep working because they compile to JavaScript.

In [ ]:
brush = alt.selection_interval()
scatter = alt.Chart(g07).mark_circle(size=60).encode(
    x=alt.X("gdpPercap:Q").scale(type="log").title("GDP per capita"),
    y=alt.Y("lifeExp:Q").scale(zero=False).title("Life expectancy"),
    color=alt.when(brush).then("continent:N").otherwise(alt.value("lightgray")),
    tooltip=["country", "lifeExp"],
).add_params(brush).properties(width=380, height=300)
bars = alt.Chart(g07).mark_bar().encode(
    x="count():Q", y=alt.Y("continent:N").title(None), color="continent:N",
).transform_filter(brush).properties(width=200, height=300)
linked = scatter | bars

linked.save("exports/linked_cdn.html")
linked.save("exports/linked_offline.html", inline=True)
print("cdn:    ", size("exports/linked_cdn.html"))
print("offline:", size("exports/linked_offline.html"))

## 3. ⚠️ Your data travels with the file

Every interactive HTML export **contains the data**, not just the picture: anyone can open the file and read it. Altair embeds every column of the DataFrame you pass, even ones the chart doesn't use.

In [ ]:
html = open("exports/linked_cdn.html").read()
print("iso_num column embedded?", '"iso_num"' in html)

Fix: pass only the columns the chart needs (and aggregate when you can). Never export confidential or identifiable data to HTML without checking.

In [ ]:
slim = g07[["country", "continent", "gdpPercap", "lifeExp"]]
linked_slim = scatter.properties(data=slim) | bars.properties(data=slim)
linked_slim.save("exports/linked_slim.html")
print("iso_num column embedded?", '"iso_num"' in open("exports/linked_slim.html").read())
print("size:", size("exports/linked_slim.html"))

## 4. HoloViews, Bokeh, and Panel -> HTML

`hvplot.save` writes a Bokeh chart. Panel's `.save(..., embed=True)` goes further: it **pre-computes** the output for every widget value and stores them in the file, so simple widget-driven apps work without a server (keep the number of widget states small).

In [ ]:
import hvplot.pandas  # noqa: F401
import panel as pn

pn.extension()

plot = g07.hvplot.scatter("gdpPercap", "lifeExp", by="continent", logx=True,
                          hover_cols=["country"], width=550, height=350)
hvplot.save(plot, "exports/hvplot_scatter.html")

continent = pn.widgets.Select(name="Continent", options=sorted(g07.continent.unique()))
def by_continent(c):
    d = g07[g07.continent == c]
    return d.sort_values("lifeExp").hvplot.barh("country", "lifeExp", height=600, width=500,
                                                title=f"Life expectancy, {c} 2007")
app = pn.Column(continent, pn.panel(pn.bind(by_continent, continent), width=500, height=600))
app.save("exports/panel_embedded.html", embed=True)
print("hvplot:", size("exports/hvplot_scatter.html"), "| panel (embedded states):",
      size("exports/panel_embedded.html"))

*Datashader-backed plots can't be embedded this way: re-aggregating on zoom needs Python running.*

## 5. A whole notebook as a report

```bash
jupyter nbconvert --to html 01_plotly.ipynb
# hide the code cells and keep only outputs:
jupyter nbconvert --to html --no-input 01_plotly.ipynb
```

Plotly and Altair outputs stay interactive in the exported page. Quarto (`quarto render`) does the same with more control over layout and can publish websites.

## 6. Apps: when you need a server

| Option | Supports | Notes |
| --- | --- | --- |
| [Streamlit Community Cloud](https://docs.streamlit.io/deploy/streamlit-community-cloud) | Streamlit | Free; deploys from a GitHub repo with `requirements.txt`. Apps sleep after 12 h without traffic; roughly 1–3 GB memory. |
| [Hugging Face Spaces](https://huggingface.co/docs/hub/spaces-sdks-streamlit) | Streamlit, Gradio, any Docker app (Dash, Panel) | For Streamlit, choose the **Docker** SDK with the Streamlit template (the built-in Streamlit SDK is deprecated). |
| Posit Connect | Streamlit, Dash, Panel, Bokeh, Shiny | Commercial; many universities and companies run one. Ask IT. |
| Your own server or cloud VM | Anything | `docker run`, or behind a reverse proxy: Dash with `gunicorn app:server`, Panel with `panel serve`, Streamlit with `streamlit run`. |
| WebAssembly, static hosting (GitHub Pages) | Streamlit via stlite; Panel via `panel convert app.py --to pyodide-worker` | No server at all; slower first load; not every package works. |

**Before you deploy:** pin versions in `requirements.txt`; keep secrets out of the repo (`st.secrets`, environment variables); check data size and who may see it; decide whether you need a login.

---

### ✏️ Exercise: Make a shareable report

Build **one** HTML file, `exports/my_report.html`, containing:

1. a title and a sentence of explanation,
2. your linked tips chart from notebook 02 (or the `linked` chart above), and
3. a Plotly chart of your choice.

Hints: `fig.to_html(full_html=False, include_plotlyjs="cdn")` gives a Plotly `<div>`. For Altair, `chart.to_json()` gives the spec; render it with vega-embed, as in the starter template. Then open the file in your browser and check that both charts are interactive.

In [ ]:
TEMPLATE = """<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>{title}</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@7"></script>
  <style>body {{ font-family: sans-serif; max-width: 960px; margin: 2em auto; }}</style>
</head>
<body>
  <h1>{title}</h1>
  <p>{intro}</p>
  <h2>Linked view</h2>
  <div id="altair-chart"></div>
  <script>vegaEmbed("#altair-chart", {altair_spec});</script>
  <h2>Plotly chart</h2>
  {plotly_div}
</body>
</html>
"""

In [ ]:
# Build your Altair chart and Plotly figure, then:
# page = TEMPLATE.format(title=..., intro=..., altair_spec=chart.to_json(),
#                        plotly_div=fig.to_html(full_html=False, include_plotlyjs="cdn"))
# with open("exports/my_report.html", "w") as f:
#     f.write(page)

## Where to go next

- **Plotly:** [plotly.com/python](https://plotly.com/python/) (the gallery is the fastest way to learn)
- **Altair:** [altair-viz.github.io](https://altair-viz.github.io/) (see "Interactive Charts" in the user guide and the example gallery)
- **Streamlit:** [docs.streamlit.io](https://docs.streamlit.io/) (start with "Get started" and the app gallery)
- **Dash:** [dash.plotly.com](https://dash.plotly.com/) (the tutorial's "Basic Callbacks" chapter)
- **HoloViz** (HoloViews, hvPlot, Panel, Datashader): [holoviz.org](https://holoviz.org/) (the tutorial ties them together)